# MirrorTopology Step 1 — Phase D4C-0b：D-4 較正の pseudo 観測列（n_pseudo＝2000・m＝1）の生成と封印（notebook v0.2）
rules §9.2 の isotropic pseudo 観測（固定 PR3 等方共分散の principal root＝D-2 の ref_matched root；独立 stream）を，**専用 pseudo 表** `d/d4_pseudo_table.json`（purpose `pseudo`（400）・group 5001・batch 0・m＝1・K＝2000；D-2 CRN 表は不変・RNG key 領域は既存の全 scope と非重複）から `d2_rng.generate_from_latent` で生成し，**全 family が共有する順序付き対 (T1_p, T2_p)**（p＝0…1999；並べ替え・行除去なし）として `d4_pseudo_columns.npz` に保存，保存 file を再読込みして列 identity（配列 SHA・paired SHA・dtype・UID 端点）を再検証した上で record（`d4_pseudo_record.json`）を publication（内容検証付き）する。target の commitment は本 notebook では計算しない（`calibration_first.commit_target(t_target, nonce)` を著者が別途実行し，commitment だけを渡す：nonce は notebook／source／log に残さない）。較正そのもの（family partial／combiner・Wilson 上端）は行わない。**標準 CPU runtime**（sandbox 自己試験 n＝50：peak RSS ≈0.3 GB・3 s；formal n＝2000 の時間は実測で記録）。入力は frozen checkout のみ（Drive の入力はない）。

**attempt cell の規約（D-3c notebook v0.3 と同一；R-D3C1-A/launch-boundary・B/output-anchor）**：attempt cell は最初に **lock の出力先（anchor）** を検証し，record はすべて anchor にだけ書く。設定・lock file・live source（実 HEAD・clean tree・inventory／script／pins／pseudo 表の file SHA・engine 版）の検査は 1 つの検査関数にまとめ，**初期 record の後**と**子 process 起動の直前**の 2 回呼ぶ（staging はない）。attempt ID を発行し，子 process 起動の前に「未完了・非成功」の initial record を `d4_pseudo_final_record.json` に書く（前回の final record は `superseded_final_record_before_<attempt>.json` に退避し，今回の証拠として再利用しない）。`pseudo_pass` は rc 0 ∧ `PSEUDO_PASS` ∧ record の shape 正常 ∧ attempt／lock／source への束縛一致 ∧ `columns.file`（`d4_pseudo_columns.npz`）の SHA／bytes 一致 ∧ n＝2000・m＝1 のときだけ True。

**v0.2（監査 R-D4C0-A／B）**：(A) attempt cell の成功 record 検査を閉じる——schema 厳密一致・`stage=='complete'`・`failures==[]`・`selftest is False`・正式 profile・**REQUIRED の集合（10 名；lock に束縛した script の `REQUIRED` 定数を AST で抽出し notebook 定数と一致を要求）と `required_inventory`・`gates` の key 集合・全値 `is True`・`required_all_true is True`**・pseudo 表 SHA＝lock・`generation.n==2000`・列 identity n＝2000／m＝1・NPZ の SHA／bytes。(B) preflight cell は **最初の mkdir／lock publication より前に出力先を検査**する（symlink・非 directory・source／Phase C／Drive root との交差・launcher 所有外の file を含む既存 directory を拒否；拒否先には何も書かない）；attempt cell も同じ規則（同じ関数）で anchor を再検査する。


In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the execution target (must contain d/d4_pseudo.py and d/d4_pseudo_table.json)>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'


In [ ]:
# --- 1. fresh scratch checkout; inventory + script + pins + pseudo table bound; the LOCK (published, hashed). No Drive input is used.
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256)
SCRATCH='/content/d4_pseudo_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
MT=SCRATCH; PHASEB=f'{SCRATCH}/engine/phaseB'; PHASEC=f'{SCRATCH}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d4_pseudo.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d4_pseudo.py']
TABLE=f'{PHASEB}/d/d4_pseudo_table.json'; PINS=f'{PHASEB}/d/d3_pins.json'; assert sha(TABLE)==inv['d_sha256']['d/d4_pseudo_table.json'] and sha(PINS)==inv['d_sha256']['d/d3_pins.json']
pins=json.load(open(PINS)); table=json.load(open(TABLE)); assert table['table_sha256']==pins['d4_pseudo_table_sha256'] and table['n_pseudo']==2000 and table['m']==1
ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl','psutil'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
import psutil; vm=psutil.virtual_memory(); RAM_INFO=dict(total_bytes=vm.total, available_bytes=vm.available); print('RAM total GB', round(vm.total/1e9,1), 'available GB', round(vm.available/1e9,1))
OUT='/content/d4_pseudo'
# OUTPUT-ANCHOR (R-D4C0-B; the lines from here to the end of this cell are re-used verbatim by the contract test): the output directory is validated BEFORE the first mkdir / lock publication
_in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
OWNED=re.compile(r'(d4_pseudo_lock\.json|d4_pseudo_final_record\.json|superseded_final_record_before_[0-9TZ]+_[0-9a-f]{10}\.json|run_[0-9TZ]+_[0-9a-f]{10}|d4_pseudo_(stdout|stderr)_[0-9TZ]+_[0-9a-f]{10}\.txt)')
def _protected_roots(): return [MT, PHASEB, PHASEC, '/content/drive']
def _safe_output_anchor(path, protected):
    # the ONE output policy (preflight and attempt): absolute path whose final component is not a symlink; disjoint from every protected root (source, Phase C, Drive);
    # non-existent, or an existing real directory holding ONLY files owned by this launcher (lock, attempt records, run_* directories, stdout/stderr) and no symlinks. Nothing is written here.
    ap=os.path.abspath(path)
    if os.path.islink(ap) or os.path.realpath(ap)!=os.path.join(os.path.realpath(os.path.dirname(ap)), os.path.basename(ap)): raise RuntimeError(f'unsafe output: {path!r} is a symbolic link; nothing written')
    clash=[q for q in protected if _in(ap,q) or _in(q,ap)]
    if clash: raise RuntimeError(f'unsafe output: {path!r} overlaps a protected root {clash}; nothing written')
    if os.path.lexists(ap):
        if not os.path.isdir(ap): raise RuntimeError(f'unsafe output: {path!r} exists and is not a directory; nothing written')
        foreign=[e for e in sorted(os.listdir(ap)) if not OWNED.fullmatch(e) or os.path.islink(os.path.join(ap,e))]
        if foreign: raise RuntimeError(f'unsafe output: {path!r} holds entries not owned by this launcher {foreign[:5]}; nothing written')
    return ap
OUT=_safe_output_anchor(OUT, _protected_roots()); os.makedirs(OUT, exist_ok=True)      # attempts are separated inside OUT by their attempt id (nothing is deleted here)
# LOCK-BUILD
def _publish(path, doc):
    data=json.dumps(doc, indent=1, ensure_ascii=False, default=str).encode('utf-8'); tmp=path+'.tmp'; open(tmp,'wb').write(data); os.replace(tmp, path); assert open(path,'rb').read()==data, path; return hashlib.sha256(data).hexdigest()
lock=dict(schema='d4_pseudo_launcher_lock_v1', commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv['d_sha256']['d/d4_pseudo.py'], table_file_sha256=sha(TABLE), table_sha256=table['table_sha256'], pins_sha256=sha(PINS), engine=inv['engine_version'],
          mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT, locked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
LOCK_PATH=f'{OUT}/d4_pseudo_lock.json'; LOCK_SHA256=_publish(LOCK_PATH, lock); print('lock', LOCK_SHA256); print(lock)


In [ ]:
# --- 2. ONE ATTEMPT (D-3c v0.3 pattern; R-D3C1-A/launch-boundary + R-D3C1-B/output-anchor; R-D4DESIGN-G): (0) the SAFE anchored output is taken from the LOCK and validated BEFORE anything is written;
#        (1) attempt id + initial NON-success record at the anchor; (2) ONE verifier (configuration, lock file, LIVE source) runs after the initial record; (3) no staging (no Drive input);
#        (4) the SAME verifier runs again IMMEDIATELY before subprocess.run (no child process on any mismatch); (5) record validation (trusted REQUIRED inventory) + binding + published-columns content check (formal n = 2000); final record at the anchor only
t0=time.time(); _in=lambda p,q: os.path.realpath(p)==os.path.realpath(q) or os.path.realpath(p).startswith(os.path.realpath(q).rstrip('/')+'/')
# 0. output anchor: from the in-memory lock (NOT the current OUT); re-validated with the SAME output policy as the preflight (_safe_output_anchor: no symlink, disjoint from the locked AND current
#    protected roots (source, Phase C, Drive), existing real directory with launcher-owned entries only) and holding the published lock file; on refusal nothing is written anywhere
ANCHOR=lock['out']
try:
    if '_safe_output_anchor' not in globals(): raise RuntimeError('preflight cell not run')
    _safe_output_anchor(ANCHOR, [lock['mt'], lock['phaseb'], lock['phasec'], '/content/drive', *_protected_roots()])
    if not os.path.isdir(ANCHOR) or LOCK_PATH!=f'{ANCHOR}/d4_pseudo_lock.json' or not os.path.isfile(LOCK_PATH): raise RuntimeError('lock path not inside the anchor')      # the lock file's CONTENT is verified by the verifier below (a FAIL record at this anchor)
except Exception as _anchor_error: print(f'ATTEMPT NOT STARTED: the locked output {ANCHOR!r} is not a safe anchored directory ({_anchor_error}); nothing was written', file=sys.stderr); raise RuntimeError('unsafe output anchor; nothing written')
ATTEMPT=time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())+'_'+hashlib.sha256(f'{time.time_ns()}|D4PSEUDO|{REPO_COMMIT}|{LOCK_SHA256}'.encode()).hexdigest()[:10]
FIN=f'{ANCHOR}/d4_pseudo_final_record.json'; RUN=f'{ANCHOR}/run_{ATTEMPT}'; pseudo_pass=False; rc=None; v=None
REC=dict(schema='d4_pseudo_launcher_final_record_v2', attempt_id=ATTEMPT, started_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), lock_sha256=LOCK_SHA256, lock=lock, output_anchor=ANCHOR, ram=globals().get('RAM_INFO'), stage='started', pseudo_pass=False, exit_code=None, PSEUDO_PASS=None, launcher_fallback=False, gates_ok=None, bindings_ok=None, evidence_ok=None, failures=[], exception=None, run_dir=RUN, bindings={}, seconds_total=None, finished_utc=None)
if os.path.exists(FIN): prev=f'{ANCHOR}/superseded_final_record_before_{ATTEMPT}.json'; os.replace(FIN, prev); REC['superseded_previous_record']=prev      # a previous attempt's record is NEVER this attempt's evidence
_publish(FIN, REC)
def _finish(stage, failures=(), exc=None):
    REC.update(stage=stage, exception=(repr(exc) if exc is not None else None), seconds_total=round(time.time()-t0, 3), finished_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())); REC['failures']+=list(failures); REC['pseudo_pass']=bool(pseudo_pass); _publish(FIN, REC); print('attempt', ATTEMPT, '| stage', stage, '| pseudo_pass', REC['pseudo_pass'], '| failures', REC['failures'])
def _verify_locked_state(prefix):
    # the ONE verifier: current configuration == lock; lock file on disk == lock; LIVE source (actual HEAD, clean tree, file SHAs of inventory / script / pins / pseudo table,
    # the inventory's script SHA, engine version, tree layout) == lock. Called after the initial record and again immediately before the child launch.
    REC['stage']=f'{prefix}_config'; cfg=dict(commit=REPO_COMMIT, mt=MT, phaseb=PHASEB, phasec=PHASEC, script=SCRIPT, out=OUT)
    mism={k: (cfg[k], lock.get(k)) for k in cfg if cfg[k]!=lock.get(k)}
    if mism: raise RuntimeError(f'configuration differs from the lock (re-run the preflight cell): {mism}')
    if LOCK_PATH!=f'{ANCHOR}/d4_pseudo_lock.json' or sha(LOCK_PATH)!=LOCK_SHA256 or json.load(open(LOCK_PATH))!=lock: raise RuntimeError('lock file on disk differs from the lock in memory')
    REC['stage']=f'{prefix}_source'; inv_now=json.load(open(INV))
    src=dict(head=subprocess.check_output(['git','-C',MT,'rev-parse','HEAD'],text=True).strip(), clean=(subprocess.check_output(['git','-C',MT,'status','--porcelain'],text=True).strip()==''), inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), inventory_script_sha256=inv_now['d_sha256']['d/d4_pseudo.py'], table_file_sha256=sha(TABLE), pins_sha256=sha(PINS), engine=inv_now['engine_version'], checked_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    want=dict(head=lock['commit'], clean=True, inventory_sha256=lock['inventory_sha256'], script_sha256=lock['script_sha256'], inventory_script_sha256=lock['script_sha256'], table_file_sha256=lock['table_file_sha256'], pins_sha256=lock['pins_sha256'], engine=lock['engine'])
    bad=[k for k in want if src[k]!=want[k]]; REC['bindings'][f'live_source_{prefix}']=src
    if bad or PHASEB!=f'{MT}/engine/phaseB' or PHASEC!=f'{MT}/phaseC' or SCRIPT!=f'{PHASEB}/d/d4_pseudo.py': raise RuntimeError(f'live source differs from the lock ({prefix}): {bad}')
try:
    # 2. configuration / lock file / live source (editing cell 0 after the preflight, or skipping the preflight, is refused here)
    _verify_locked_state('precheck'); _publish(FIN, REC)
    # 4. the SAME verifier IMMEDIATELY before the launch: any mismatch -> FAIL record, NO child process
    _verify_locked_state('prelaunch')
    REC['stage']='launch'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',RUN,'--profile','production_official','--attempt-id',ATTEMPT,'--launcher-lock-sha256',LOCK_SHA256]
    REC['args']=args; _publish(FIN, REC); rc=subprocess.run(args,capture_output=True,text=True); REC['exit_code']=rc.returncode
    open(f'{ANCHOR}/d4_pseudo_stdout_{ATTEMPT}.txt','w').write(rc.stdout); open(f'{ANCHOR}/d4_pseudo_stderr_{ATTEMPT}.txt','w').write(rc.stderr); print(rc.stdout[-3000:]); print('rc', rc.returncode)
    # 5. semantic validation of the record (R-D4C0-A): exact schema / complete stage / empty failures / formal profile / no self-test (shape -> launcher_fallback); the TRUSTED REQUIRED inventory
    #    (notebook constant == the REQUIRED tuple extracted by AST from the lock-bound script) with every gate `is True` and required_all_true (-> gates_ok); binding to THIS attempt / lock / source /
    #    table (-> bindings_ok); the published columns file content-verified with the formal n = 2000, m = 1 identity (-> evidence_ok)
    REC['stage']='record'; vp=f'{RUN}/d4_pseudo_record.json'; shape=[]; gates_bad=[]; binding=[]; evidence=[]
    REQUIRED_EXPECTED=['G_pins_loaded','G_engine_inventory','G_script_sha','G_phaseC_members','G_env_lock','G_external_loader_sha','G_pseudo_table','G_generated','G_columns_verified','G_record_saved']
    import ast as _ast
    _req_src=[list(_ast.literal_eval(st.value)) for st in _ast.parse(open(SCRIPT).read()).body if isinstance(st,_ast.Assign) and any(isinstance(t,_ast.Name) and t.id=='REQUIRED' for t in st.targets)]
    if _req_src!=[REQUIRED_EXPECTED]: gates_bad.append(f'REQUIRED inventory of the lock-bound script differs from the trusted constant: {_req_src}')
    try: v=json.load(open(vp))
    except Exception as read_error: v=None; shape.append(f'pseudo record unreadable: {read_error!r}')
    if v is not None and not isinstance(v, dict): shape.append('pseudo record is not a dictionary'); v=None
    if v is not None:
        num=lambda x: isinstance(x,(int,float)) and not isinstance(x,bool); cols=v.get('columns') if isinstance(v.get('columns'), dict) else {}
        for k, ok in [('schema', v.get('schema')=='d4_pseudo_record_v1'), ('PSEUDO_PASS', isinstance(v.get('PSEUDO_PASS'), bool)), ('stage', v.get('stage')=='complete'), ('gates', isinstance(v.get('gates'), dict)), ('required_inventory', isinstance(v.get('required_inventory'), list)), ('required_all_true', isinstance(v.get('required_all_true'), bool)), ('failures', v.get('failures')==[]), ('seconds', num(v.get('seconds'))), ('stages_rss_mb', isinstance(v.get('stages_rss_mb'), dict)),
                      ('attempt', isinstance(v.get('attempt'), dict)), ('source', isinstance(v.get('source'), dict)), ('selftest', v.get('selftest') is False), ('profile', v.get('profile')=='production_official'), ('columns', isinstance(cols.get('identity'), dict) and isinstance(cols.get('file'), dict)), ('pseudo_table', isinstance(v.get('pseudo_table'), dict)), ('generation', isinstance(v.get('generation'), dict))]:
            if not ok: shape.append(f'pseudo record field invalid: {k}')
        g=v.get('gates') if isinstance(v.get('gates'), dict) else {}
        if sorted(g)!=sorted(REQUIRED_EXPECTED): gates_bad.append(f'gate set differs from the trusted REQUIRED inventory: {sorted(g)}')
        if v.get('required_inventory')!=REQUIRED_EXPECTED: gates_bad.append('required_inventory differs from the trusted REQUIRED inventory')
        if not all(g.get(k) is True for k in REQUIRED_EXPECTED): gates_bad.append(f'required gates not all True: {[k for k in REQUIRED_EXPECTED if g.get(k) is not True]}')
        if v.get('required_all_true') is not True: gates_bad.append('required_all_true is not True')
        a_=v.get('attempt') if isinstance(v.get('attempt'), dict) else {}; s_=v.get('source') if isinstance(v.get('source'), dict) else {}; t_=v.get('pseudo_table') if isinstance(v.get('pseudo_table'), dict) else {}; gen=v.get('generation') if isinstance(v.get('generation'), dict) else {}
        for k, ok in [('attempt_id', a_.get('attempt_id')==ATTEMPT), ('launcher_lock_sha256', a_.get('launcher_lock_sha256')==LOCK_SHA256), ('source.script_sha256', s_.get('script_sha256')==lock['script_sha256']), ('source.inventory_sha256', s_.get('inventory_sha256')==lock['inventory_sha256']), ('source.pins_sha256', s_.get('pins_sha256')==lock['pins_sha256']), ('source.engine_version', s_.get('engine_version')==lock['engine']), ('pseudo_table.table_sha256', t_.get('table_sha256')==lock['table_sha256']), ('pseudo_table.n_pseudo', t_.get('n_pseudo')==2000 and t_.get('m')==1)]:
            if not ok: binding.append(f'pseudo record not bound to this attempt / lock / table: {k}')
        fe=cols.get('file') if isinstance(cols.get('file'), dict) else {}; idn=cols.get('identity') if isinstance(cols.get('identity'), dict) else {}; fp=f"{RUN}/{fe.get('file')}" if isinstance(fe.get('file'), str) else None
        if not (fp and os.path.basename(fp)=='d4_pseudo_columns.npz' and os.path.isfile(fp) and not os.path.islink(fp) and sha(fp)==fe.get('sha256') and os.path.getsize(fp)==fe.get('bytes')): evidence.append('published columns file not content-verified: d4_pseudo_columns.npz')
        if not (idn.get('n')==2000 and idn.get('m')==1 and gen.get('n')==2000 and gen.get('m')==1 and idn.get('first_uid')==[1,400,5001,0,0] and idn.get('last_uid')==[1,400,5001,0,1999]): evidence.append(f"column identity is not the formal n=2000, m=1 ordered column ({idn.get('n')}, {idn.get('m')}, {gen.get('n')}, {idn.get('first_uid')}, {idn.get('last_uid')})")
        REC.update(PSEUDO_PASS=v.get('PSEUDO_PASS'), record_sha256=sha(vp), record_stage=v.get('stage'), record_failures=v.get('failures'), gates=v.get('gates'), stages_rss_mb=v.get('stages_rss_mb'), stages_peak_rss_mb=v.get('stages_peak_rss_mb'), seconds_script=v.get('seconds'), columns_identity=idn, columns_file=fe)
        print('rc', rc.returncode, '| stage', v.get('stage'), '| PSEUDO_PASS', v.get('PSEUDO_PASS'), '| failures', v.get('failures'), '| rss MB', v.get('stages_rss_mb'), '| peak rss MB', v.get('stages_peak_rss_mb'), '| seconds', v.get('seconds')); print('columns', idn.get('n'), idn.get('m'), idn.get('paired_sha256'))
    REC.update(launcher_fallback=bool(shape), gates_ok=(not gates_bad) if v is not None else None, bindings_ok=(not binding) if v is not None else None, evidence_ok=(not evidence) if v is not None else None, stderr_tail=rc.stderr[-2000:])
    pseudo_pass=bool(rc.returncode==0 and v is not None and v.get('PSEUDO_PASS') is True and not shape and not gates_bad and not binding and not evidence)
    _finish('complete', shape+gates_bad+binding+evidence)
except Exception as attempt_error:
    pseudo_pass=False; _finish(REC.get('stage') or 'unknown', [f'exception at stage {REC.get("stage")}: {attempt_error!r}'], attempt_error)
print('pseudo_pass', pseudo_pass)


In [ ]:
# --- 3. zip the evidence (lock, attempt records (current + superseded), pseudo record, d4_pseudo_columns.npz, stdout/stderr) for the audit and for registration
zp=f'/content/d4_pseudo_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=lock['out'])
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)